# SC203 — Task M1-1: Real-CE (13mm -> 52mm, 4x) Dataset Feasibility, Pairing & DoSSR Smoke Check
**Course:** SC203 Scientific Methods (Class 25A02) — Team 1 (`LEMMEPASS`)  
**Assigned Owner:** Member 1 — Hoàng Nguyên Anh (`25125002`, Team Lead / GPU & Integration Lead)  
**Repository & Branch:** `https://github.com/lemon4life/SC203.git` (`feat/baseline-runner`)  
**Official Reference:** *Real-CE: A Benchmark for Chinese-English Scene Text Image Super-resolution* (ICCV 2023) — [`mjq11302010044/Real-CE`](https://github.com/mjq11302010044/Real-CE)

---

### Real-CE Dataset Architecture (Focal-Length Pairs)
Unlike RealSR (`test_LR` / `test_HR`), Real-CE captures real-world Chinese-English scene text with three focal lengths:
- **`train/`** (official training split) and **`val/`** (official evaluation split)
- Inside each split: **`13mm/`** ($4\times$ LR), **`26mm/`** ($2\times$ LR), **`52mm/`** (HR ground truth), **`trans_anoon/`** (or `trans_annos/` text box & transcript files), and **`valid.txt`** (split image list).

---

### Master Cell Directory & Estimated Times

| Cell | Name | Purpose | What to Expect | Est. Time |
| :---: | :--- | :--- | :--- | :---: |
| **`CELL 01`** | **Drive Mount & Paths** | Mount Drive & set directory paths | Mounted at `/content/drive`, path verification | ~10s |
| **`CELL 02`** | **Repo Sync & Branch** | Clone or pull `feat/baseline-runner` | Fast-forwards to latest commit, prints HEAD hash | ~5s |
| **`CELL 03`** | **GPU & VRAM Check** | Verify CUDA GPU, PyTorch & VRAM | Displays GPU model (L4/T4/A100) & total VRAM | ~5s |
| **`CELL 04`** | **Stage Real-CE (Fast SSD)** | Reuse unpacked Drive/SSD or unpack zip to `/content/data_local/Real-CE` | Fast local SSD staging (~10s unpack from `.zip`) | ~10s–2m |
| **`CELL 05`** | **Structure & 4x Audit** | Audit `train`/`val`, `13mm`/`26mm`/`52mm`, `valid.txt` & pairs | Confirms `13mm <-> 52mm` 4x pairs and dimensions | ~10s |
| **`CELL 06`** | **Annotation Audit** | Inspect `trans_anoon/` / `trans_annos/` & `valid.txt` | Reports box format, English/Chinese flags, samples | ~10s |
| **`CELL 07`** | **HR OCR Benchmark** | Evaluate RapidOCR readability on `52mm` HR | Computes HR readability ceiling vs RealSR $70.5\%$ | ~30s |
| **`CELL 08`** | **DoSSR Smoke Run** | 3-image $128\times128 \to 512\times512$ DoSSR run on `13mm` | Logs peak VRAM (GiB) and runtime | ~30s |
| **`CELL 09`** | **Export Summary JSON** | Save `realce_feasibility_summary.json` | Exports summary JSON to repo & Drive | ~5s |


--- ## 1. Google Drive Mount & Path Configuration (`CELL 01`)
- **Purpose:** Mount Google Drive, establish shared team paths (`/content/drive/MyDrive/lemmepass`), and initialize runtime directories.
- **What to Expect:** Successful mount output at `/content/drive` and confirmed folder structure.
- **Estimated Time:** ~10 seconds.


In [ ]:
# CELL 01 [IN-CELL RUN] — Mount Google Drive & configure path constants
from datetime import datetime, timezone
import os
from pathlib import Path
import sys

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("[WARNING] Not running in Google Colab environment. Using local filesystem.")

# Shared Team 1 (LEMMEPASS) layout
DRIVE_ROOT = Path("/content/drive/MyDrive/lemmepass")
REPO_DIR = Path("/content/SC203")
DATA_DIR_DRIVE = DRIVE_ROOT / "data" / "Real-CE"
DATA_DIR_LOCAL = Path("/content/data_local/Real-CE")
OUTPUT_DIR = REPO_DIR / "results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BRANCH = "feat/baseline-runner"
REPO_URL = "https://github.com/lemon4life/SC203.git"

print(f"Timestamp      : {datetime.now(timezone.utc).isoformat()}")
print(f"Drive Root Path: {DRIVE_ROOT} (exists: {DRIVE_ROOT.exists()})")
print(f"Local SSD Data : {DATA_DIR_LOCAL}")
print(f"Results Output : {OUTPUT_DIR}")


--- ## 2. Repository Sync & Branch Verification (`CELL 02`)
- **Purpose:** Fast-forward repository to branch `feat/baseline-runner` to pull latest scripts and documentation.
- **What to Expect:** Prints the active git commit hash and verifies working copy cleanliness.
- **Estimated Time:** ~5 seconds.


In [ ]:
# CELL 02 [IN-CELL RUN] — Clone or fast-forward repository to feat/baseline-runner
import os
from pathlib import Path
import shutil
import subprocess

subprocess.run(['git', 'config', '--global', '--add', 'safe.directory', '*'], check=False)

def sync_repo(repo_dir: Path, repo_url: str, branch: str):
    need_fresh_clone = False
    if not (repo_dir / '.git').exists():
        need_fresh_clone = True
    else:
        print(f'Fast-forwarding existing repo at {repo_dir}...')
        try:
            subprocess.run(['git', '-C', str(repo_dir), 'remote', 'set-url', 'origin', repo_url], check=False)
            res_fetch = subprocess.run(['git', '-C', str(repo_dir), 'fetch', 'origin', branch], capture_output=True, text=True)
            if res_fetch.returncode != 0:
                print(f'[WARNING] git fetch failed. Falling back to clean clone...')
                need_fresh_clone = True
            else:
                subprocess.run(['git', '-C', str(repo_dir), 'checkout', branch], check=True)
                subprocess.run(['git', '-C', str(repo_dir), 'reset', '--hard', f'origin/{branch}'], check=True)
        except Exception as e:
            print(f'[WARNING] Error updating existing repo ({e}). Falling back to clean clone...')
            need_fresh_clone = True

    if need_fresh_clone:
        print(f'Cloning fresh copy of {repo_url} (branch: {branch})...')
        if repo_dir.exists():
            shutil.rmtree(repo_dir, ignore_errors=True)
        subprocess.run(['git', 'clone', '--branch', branch, '--single-branch', repo_url, str(repo_dir)], check=True)

    head_hash = subprocess.run(['git', '-C', str(repo_dir), 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True, check=True).stdout.strip()
    print(f'Active Git Commit: {head_hash} on branch {branch}')

sync_repo(REPO_DIR, REPO_URL, BRANCH)


--- ## 3. Hardware, CUDA & GPU VRAM Verification (`CELL 03`)
- **Purpose:** Detect available GPU accelerator (e.g., NVIDIA L4, T4, A100), record total VRAM, and reset memory statistics.
- **What to Expect:** Device name, total memory (GiB), CUDA version, and initial peak VRAM = 0.0 GiB.
- **Estimated Time:** ~5 seconds.


In [ ]:
# CELL 03 [IN-CELL RUN] — Check CUDA GPU, VRAM capacity, and PyTorch environment
import torch

cuda_ok = torch.cuda.is_available()
if not cuda_ok:
    print("[WARNING] CUDA is NOT available. Running on CPU (DoSSR GPU smoke test will be skipped).")
    gpu_name = "CPU"
    total_mem_gib = 0.0
else:
    gpu_name = torch.cuda.get_device_name(0)
    total_mem_bytes = torch.cuda.get_device_properties(0).total_memory
    total_mem_gib = round(total_mem_bytes / (1024**3), 2)
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    print(f"GPU Accelerator : {gpu_name}")
    print(f"Total GPU VRAM  : {total_mem_gib} GiB")
    print(f"PyTorch Version : {torch.__version__}")
    print(f"CUDA Version    : {torch.version.cuda}")
    print(f"cuDNN Version   : {torch.backends.cudnn.version()}")


--- ## 4. Real-CE Dataset Smart Staging (`CELL 04`)
- **Purpose:** Avoid slow small-file writes over Google Drive FUSE by extracting `Real-CE.zip` onto local fast Colab SSD (`/content/data_local/Real-CE`, ~10 seconds). Automatically reuses already-extracted data on local SSD or Google Drive if present.
- **What to Expect:** Confirmed Real-CE root containing `train/` and `val/` with `13mm/`, `26mm/`, `52mm/`, `trans_anoon/`, and `valid.txt`.
- **Estimated Time:** ~10 seconds (if `.zip` is cached) or ~1–2 minutes (fresh `gdown`).


In [ ]:
# CELL 04 [IN-CELL RUN] — Fast SSD staging of Real-CE (with automatic Drive/zip reuse)
import shutil
import subprocess
import zipfile

sys.path.insert(0, str(REPO_DIR))
from scripts.check_realce_feasibility import resolve_realce_root

def _has_realce_splits(candidate: Path) -> bool:
    root = resolve_realce_root(candidate)
    return (root / "train" / "13mm").is_dir() and (root / "train" / "52mm").is_dir()

REALCE_GDRIVE_ID = "1d2pOgJ0e286OslzuGVsARfhW7FbQW0n-"
zip_drive_target = DRIVE_ROOT / "data" / "Real-CE.zip"
zip_local_target = Path("/content/Real-CE.zip")
(DRIVE_ROOT / "data").mkdir(parents=True, exist_ok=True)

if _has_realce_splits(DATA_DIR_LOCAL):
    DATA_DIR = resolve_realce_root(DATA_DIR_LOCAL)
    print(f"Reusing already unpacked local SSD dataset: {DATA_DIR}")
elif zip_local_target.is_file() or zip_drive_target.is_file():
    target_zip = zip_local_target if zip_local_target.is_file() else zip_drive_target
    if target_zip == zip_drive_target and not zip_local_target.is_file():
        print(f"Copying cached archive from Drive to local SSD: {zip_drive_target}")
        shutil.copy2(zip_drive_target, zip_local_target)
        target_zip = zip_local_target
    elif not zip_drive_target.is_file():
        try:
            shutil.copy2(zip_local_target, zip_drive_target)
            print(f"Backed up archive to Drive: {zip_drive_target}")
        except Exception as exc:
            print(f"[NOTE] Drive backup skipped: {exc}")
    DATA_DIR_LOCAL.mkdir(parents=True, exist_ok=True)
    print(f"Extracting {target_zip} to fast local SSD ({DATA_DIR_LOCAL})...")
    rc = subprocess.run(["unzip", "-q", "-o", str(target_zip), "-d", str(DATA_DIR_LOCAL)], check=False).returncode
    if rc != 0:
        with zipfile.ZipFile(target_zip, "r") as zf:
            zf.extractall(DATA_DIR_LOCAL)
    DATA_DIR = resolve_realce_root(DATA_DIR_LOCAL)
elif _has_realce_splits(DATA_DIR_DRIVE):
    DATA_DIR = resolve_realce_root(DATA_DIR_DRIVE)
    print(f"Reusing already unpacked Google Drive dataset: {DATA_DIR}")
else:
    print(f"Downloading Real-CE.zip via gdown (ID: {REALCE_GDRIVE_ID})...")
    subprocess.run(["pip", "install", "-q", "gdown"], check=True)
    import gdown
    gdown.download(id=REALCE_GDRIVE_ID, output=str(zip_local_target), quiet=False)
    try:
        shutil.copy2(zip_local_target, zip_drive_target)
        print(f"Backed up single archive to Drive: {zip_drive_target}")
    except Exception as exc:
        print(f"[NOTE] Drive backup skipped: {exc}")
    DATA_DIR_LOCAL.mkdir(parents=True, exist_ok=True)
    print(f"Extracting {zip_local_target} to fast local SSD ({DATA_DIR_LOCAL})...")
    rc = subprocess.run(["unzip", "-q", "-o", str(zip_local_target), "-d", str(DATA_DIR_LOCAL)], check=False).returncode
    if rc != 0:
        with zipfile.ZipFile(zip_local_target, "r") as zf:
            zf.extractall(DATA_DIR_LOCAL)
    DATA_DIR = resolve_realce_root(DATA_DIR_LOCAL)

subdirs = sorted(p.name for p in DATA_DIR.iterdir() if p.is_dir())
print(f"Resolved DATA_DIR        : {DATA_DIR}")
print(f"Top-level subdirectories : {subdirs}")


--- ## 5. Split Structure, Focal Lengths (`13mm`/`26mm`/`52mm`) & 4x Scale Audit (`CELL 05`)
- **Purpose:** Audit `train/` and `val/` counts across `13mm` ($4\times$ LR), `26mm` ($2\times$ LR), `52mm` (HR), `trans_anoon/`, and `valid.txt`, and verify matched `13mm <-> 52mm` image dimensions.
- **What to Expect:** Per-split pair breakdown and dimension/ratio verification on 5 sample pairs.
- **Estimated Time:** ~10 seconds.


In [ ]:
# CELL 05 [IN-CELL RUN] — Audit train/val, 13mm/26mm/52mm focal folders, valid.txt & 4x pairing
sys.path.insert(0, str(REPO_DIR))
from scripts.check_realce_feasibility import (
    audit_dataset_structure,
    find_lr_hr_pairs,
    inspect_image_pair,
    resolve_realce_root,
)

DATA_DIR = resolve_realce_root(DATA_DIR)
struct_res = audit_dataset_structure(DATA_DIR)
print(f'Audit Status   : {struct_res["status"]}')
print(f'Resolved Root  : {struct_res["resolved_root"]}')
print(f'Total 4x Pairs : {struct_res["total_pairs"]} (13mm <-> 52mm)')
print(f'Total 13mm LR  : {struct_res.get("total_lr_found", 0)}')
print(f'Total 52mm HR  : {struct_res.get("total_hr_found", 0)}')

print('\n--- Split Breakdown ---')
for s_name, s_info in struct_res.get('splits', {}).items():
    print(
        f'Split [{s_name}]: 13mm(4x)={s_info.get("lr_13mm_x4_count", 0)} | '
        f'26mm(2x)={s_info.get("lr_26mm_x2_count", 0)} | '
        f'52mm(HR)={s_info.get("hr_52mm_count", 0)} | '
        f'matched_4x={s_info.get("matched_4x_pairs", 0)} | '
        f'valid.txt={s_info.get("valid_txt_entries", 0)} | '
        f'annos={s_info.get("trans_annos_file_count", 0)} ({s_info.get("trans_annos_dirs")})'
    )

lr_files, hr_files = find_lr_hr_pairs(DATA_DIR, scale='x4')
print(f'\nMatched 4x pairs returned: {len(lr_files)}')

print('\n--- 4x Sample Dimension Check (First 5 Matched 13mm -> 52mm Pairs) ---')
sample_checks = []
for lr_p, hr_p in zip(lr_files[:5], hr_files[:5]):
    chk = inspect_image_pair(lr_p, hr_p, root_dir=DATA_DIR)
    sample_checks.append(chk)
    print(
        f'Pair: {chk["lr_file"]} {chk["lr_size"]} -> {chk["hr_file"]} {chk["hr_size"]} | '
        f'Scale=({chk["scale_w"]}x, {chk["scale_h"]}x) | Exact 4x: {chk["is_exact_4x"]} | Same canvas: {chk["is_same_canvas"]}'
    )


--- ## 6. Text Annotation (`trans_anoon/` / `trans_annos/`) & `valid.txt` Audit (`CELL 06`)
- **Purpose:** Inspect transcription annotation files inside `trans_anoon/` / `trans_annos/` (separate from `valid.txt`), coordinate schema, and English vs. Chinese language coverage.
- **What to Expect:** Detected annotation directory, box format, sample annotation lines, and `valid.txt` sample lines.
- **Estimated Time:** ~10 seconds.


In [ ]:
# CELL 06 [IN-CELL RUN] — Audit trans_anoon/ / trans_annos/ & valid.txt
from scripts.check_realce_feasibility import audit_annotations

annot_res = audit_annotations(DATA_DIR)
print(f"Has Annotations  : {annot_res.get('has_annotations')}")
print(f"Annotation Dirs  : {annot_res.get('annotation_dirs')}")
print(f"Annotation Files : {annot_res.get('annotation_file_count')}")
print(f"Sample Anno File : {annot_res.get('sample_filename')}")
print(f"Detected Format  : {annot_res.get('detected_box_type')}")
print(f"Has English Text : {annot_res.get('has_english_text')}")
print(f"Has Chinese Text : {annot_res.get('has_chinese_text')}")

print("\n--- Sample Raw Annotation Excerpt ---")
for line in annot_res.get("sample_excerpt", []):
    print(f"  {line}")

print(f"\nvalid.txt files  : {annot_res.get('valid_txt_files')}")
for line in annot_res.get("valid_txt_sample", []):
    print(f"  [valid.txt] {line}")


--- ## 7. High-Resolution (`52mm` HR) OCR Readability Benchmark (`CELL 07`)
- **Purpose:** Evaluate RapidOCR readability on a sample of 30 Real-CE `52mm` HR images.
- **What to Expect:** Computes Real-CE's HR readability rate (contrasted with RealSR's $70.5\%$).
- **Estimated Time:** ~30 seconds.


In [ ]:
# CELL 07 [IN-CELL RUN] — Benchmark RapidOCR on Real-CE 52mm HR images
import subprocess
subprocess.run(["pip", "install", "-q", "rapidocr_onnxruntime"], check=True)

from scripts.check_realce_feasibility import run_sample_hr_ocr

print("Running RapidOCR on sample 52mm HR images...")
ocr_res = run_sample_hr_ocr(hr_files, max_samples=30)
print(f"OCR Status   : {ocr_res['status']}")
if ocr_res.get("status") == "COMPLETED":
    print(f"HR Tested    : {ocr_res['samples_tested']} images")
    print(f"HR Readable  : {ocr_res['readable_count']} ({ocr_res['readable_percentage']})")
    print("\n--- Sample Recognized Transcripts ---")
    for item in ocr_res.get("sample_predictions", []):
        print(f"  [{item.get('file')}]: \"{item.get('text')}\"")
else:
    print(f"Message: {ocr_res.get('message')}")


--- ## 8. DoSSR 3-Image Inference Smoke Test & VRAM Logging (`CELL 08`)
- **Purpose:** Stage 3 `13mm` LR crops as standardized `128x128` PNGs and run DoSSR (`steps: 5`, `seed 42`, `fp16`, `disable_sdpa: true`) -> `512x512`.
- **What to Expect:** Logs peak CUDA memory (`peak_vram_allocated_gib`) and runtime (~20–30s total).
- **Estimated Time:** ~30–60 seconds.


In [ ]:
# CELL 08 [IN-CELL RUN] — Run DoSSR 5-step smoke test on 3 standardized 128x128 crops from 13mm
from PIL import Image
import shutil
import time
import yaml

if not cuda_ok:
    print('[SKIPPED] CUDA GPU not available; skipping DoSSR inference test.')
else:
    smoke_stage_dir = Path('/content/realce_smoke_lr')
    if smoke_stage_dir.exists():
        shutil.rmtree(smoke_stage_dir)
    smoke_stage_dir.mkdir(parents=True, exist_ok=True)
    out_smoke_dir = Path('/content/outputs/realce_smoke_dossr')
    out_smoke_dir.mkdir(parents=True, exist_ok=True)

    candidates = lr_files[:3] if 'lr_files' in globals() and lr_files else []
    for idx, p in enumerate(candidates):
        with Image.open(p) as im_raw:
            im = im_raw.convert('RGB')
            w, h = im.size
            if w >= 128 and h >= 128:
                left = (w - 128) // 2
                top = (h - 128) // 2
                crop = im.crop((left, top, left + 128, top + 128))
            else:
                side = min(w, h)
                left = (w - side) // 2
                top = (h - side) // 2
                crop = im.crop((left, top, left + side, top + side)).resize((128, 128), Image.Resampling.BICUBIC)
            crop.save(smoke_stage_dir / f'{p.stem}.png')

    staged_count = len(list(smoke_stage_dir.glob('*.png')))
    print(f'Staged {staged_count} standardized 128x128 13mm crops into {smoke_stage_dir}.')

    if staged_count == 0:
        print('[WARNING] 0 images were staged. Verify CELL 04 & CELL 05 found 13mm pairs.')
    else:
        ckpt_path = DRIVE_ROOT / 'weights' / 'dossr_default.ckpt'
        model_cfg = REPO_DIR / 'external' / 'DOSSR' / 'configs' / 'model' / 'cldm_v21.yaml'
        inference_script = REPO_DIR / 'external' / 'DOSSR' / 'scripts' / 'inference.py'
        dossr_root = REPO_DIR / 'external' / 'DOSSR'
        if not model_cfg.exists():
            print('Missing external/DOSSR config; bootstrapping via scripts/colab_setup.py...')
            subprocess.run([sys.executable, 'scripts/colab_setup.py'], cwd=str(REPO_DIR), check=True)
        log_dir = DRIVE_ROOT / 'logs'
        log_dir.mkdir(parents=True, exist_ok=True)

        smoke_yaml = Path('/content/baseline_realce_smoke.yaml')
        smoke_config = {
            'seed': 42,
            'input_dir': str(smoke_stage_dir),
            'output_root': str(out_smoke_dir),
            'checkpoint': str(ckpt_path),
            'model_config': str(model_cfg),
            'inference_script': str(inference_script),
            'dossr_root': str(dossr_root),
            'log_dir': str(log_dir),
            'steps': 5,
            'solver_order': 3,
            't1_fraction': 0.5,
            'device': 'cuda',
            'precision': 'fp16',
            'sr_scale': 4,
            'color_fix_type': 'wavelet',
            'preprocessor': 'RealESRNet',
            'prompt_mode': 'default',
            'disable_sdpa': True,
        }
        with open(smoke_yaml, 'w', encoding='utf-8') as f:
            yaml.safe_dump(smoke_config, f)

        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

        if ckpt_path.exists() and model_cfg.exists():
            t0 = time.perf_counter()
            cmd = [
                sys.executable, 'src/run_dossr.py',
                '--config', str(smoke_yaml),
                '--fp16',
                '--disable-sdpa',
            ]
            print(f'Executing: {" ".join(cmd)}')
            subprocess.run(cmd, cwd=str(REPO_DIR), check=True)
            runtime_s = round(time.perf_counter() - t0, 2)
            peak_vram_gib = round(torch.cuda.max_memory_allocated() / (1024**3), 3)
            print(f'\nDoSSR Smoke Test Completed in {runtime_s}s!')
            print(f'Peak VRAM Allocated: {peak_vram_gib} GiB')
        else:
            missing = []
            if not ckpt_path.exists(): missing.append(f'Checkpoint ({ckpt_path})')
            if not model_cfg.exists(): missing.append(f'Model Config ({model_cfg})')
            print(f'[NOTE] Missing: {", ".join(missing)}. Skipping forward pass.')


--- ## 9. Feasibility Summary JSON Export & Mentor Decision Brief (`CELL 09`)
- **Purpose:** Consolidate metrics from Cells 04–08 into `results/realce_feasibility_summary.json`, copy to Google Drive, and print an executive summary.
- **What to Expect:** Structured JSON saved to both local repo and Google Drive.
- **Estimated Time:** ~5 seconds.


In [ ]:
# CELL 09 [IN-CELL RUN] — Export realce_feasibility_summary.json & print executive brief
import json

summary_file_repo = REPO_DIR / "results" / "realce_feasibility_summary.json"
summary_file_drive = DRIVE_ROOT / "results" / "realce_feasibility_summary.json"
summary_file_drive.parent.mkdir(parents=True, exist_ok=True)

report_data = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "dataset": "Real-CE (ICCV 2023)",
    "official_url": "https://github.com/mjq11302010044/Real-CE",
    "hardware": {
        "gpu_name": gpu_name if 'gpu_name' in globals() else "Unknown",
        "total_vram_gib": total_mem_gib if 'total_mem_gib' in globals() else 0.0,
    },
    "structure_audit": struct_res if 'struct_res' in globals() else {},
    "dimension_checks": sample_checks if 'sample_checks' in globals() else [],
    "annotation_audit": annot_res if 'annot_res' in globals() else {},
    "hr_ocr_ceiling": ocr_res if 'ocr_res' in globals() else {},
}

with open(summary_file_repo, "w", encoding="utf-8") as f:
    json.dump(report_data, f, indent=2)

try:
    shutil.copy2(summary_file_repo, summary_file_drive)
    print(f"Summary JSON backed up to Drive : {summary_file_drive}")
except Exception as e:
    print(f"Could not copy to Drive ({e}); saved locally.")

print(f"Summary JSON saved to repository: {summary_file_repo}")

print("\n======================================================================")
print("EXECUTIVE DECISION BRIEF FOR MENTOR (DR. THUẬN):")
print("======================================================================")
print(f"1. Dataset & Splits : {struct_res.get('total_pairs', 'N/A')} matched 4x (13mm -> 52mm) pairs")
for s_name, s_info in struct_res.get('splits', {}).items():
    print(f"   - {s_name}: 13mm={s_info.get('lr_13mm_x4_count', 0)}, 26mm={s_info.get('lr_26mm_x2_count', 0)}, 52mm={s_info.get('hr_52mm_count', 0)}, valid.txt={s_info.get('valid_txt_entries', 0)}, annos={s_info.get('trans_annos_file_count', 0)}")
print(f"2. Text Annotations : Dirs={annot_res.get('annotation_dirs')}, Format={annot_res.get('detected_box_type', 'N/A')}, EN={annot_res.get('has_english_text')}, ZH={annot_res.get('has_chinese_text')}")
if 'ocr_res' in globals() and ocr_res.get('status') == 'COMPLETED':
    print(f"3. 52mm HR Ceiling  : {ocr_res.get('readable_percentage')} readable on HR (vs 70.5% on RealSR).")
print("4. Conclusion       : Real-CE train (13mm -> 52mm) ready for scene-level split & LoRA 2/3 pilot.")
print("======================================================================")
